<a href="https://colab.research.google.com/github/valentinaslisser/mlandmlops/blob/main/werkgroep_hierarchical_RL_TA2_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Werkgroep: Hiërarchische Reinforcement Learning

## Inleiding Computationele Psychologie

Welkom bij het werkcollege over hiërarchische RL. In Laptop College 2 heb je Q-learning toegepast op een kleine actie-ruimte, met directe feedback na elke keuze. In de echte wereld is gedrag complexer. Complexe doelen vragen om *sequenties* van acties, en feedback komt pas veel later. Dat is deel van het **credit assignment probleem** — een van de centrale uitdagingen in RL.

In dit werkcollege ga je ervaren waarom een simpele Q-learner langzaam leert in zulke domeinen, en hoe **hiërarchie** (Sutton, Precup & Singh, 1999) het probleem drastisch verkleint. Je werkt met een versie van de klassieke Taxi-taak (Dietterich, 2000) waarbij de taxi naast mensen moet ophalen en brengen ook brandstof moet beheren.

We introduceren de brandstof management als een minimaal model voor *homeo- & allostase* (Sterling, 2012).

**Werk in groepjes van 4** met deze rolverdeling:
* **driver** — typt en deelt het scherm
* **navigators** — lezen mee, controleert de code en bewaken de tijd


**Referenties**:
* Sutton, R.S., Precup, D., & Singh, S. (1999). Between MDPs and semi-MDPs. *Artificial Intelligence*, 112(1–2), 181–211.
* Dietterich, T.G. (2000). Hierarchical reinforcement learning with the MAXQ value function decomposition. *JAIR*, 13, 227–303.
* Sterling, P. (2012). Allostasis: A model of predictive regulation. *Physiology & Behavior*, 106(1), 5–15.
* Keramati, M., & Gutkin, B. (2014). Homeostatic reinforcement learning for integrating reward collection and physiological stability. *eLife*, 3, e04811.
* Tolman, E.C. (1948). Cognitive maps in rats and men. *Psychological Review*, 55(4), 189–208.


---

## 0. De Taxi-Fuel omgeving

We werken met een 5×5 grid met vier passagier/bestemming locaties (R, G, Y, B) en één tankstation (F):

```
    +---------+
    |R: | : :G|
    | : | : : |
    | : :F: : |
    | | : | : |
    |Y| : |B: |
    +---------+
```

De taxi krijgt elke keer een nieuwe klant met een willekeurige pickup-locatie en bestemming. Regels:

* **Elke actie kost 1 brandstof** én –1 reward (ook illegale acties zoals tanken op de verkeerde plek). Dit motiveert de agent om efficiënt te zijn.
* Succesvolle dropoff op de juiste locatie: **+50** reward.
* Tanken op F (alleen zonder passagier): brandstof terug naar vol, kosten **–10**.
* Als brandstof op nul raakt wordt de taxi gesleept (**–50**); als er een passagier aan boord was gaat die ook verloren (**–50** extra). De episode gaat daarna door.

**Een episode duurt 240 tijdstappen.** Een goed beleid haalt 10–15 klanten in die tijd.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

from taxi_fuel_env_pretrain import (
    TaxiFuelEnv, get_default_options, pretrain_go_to_options,
    LearnedGoToOption,
    LOC_R, LOC_G, LOC_Y, LOC_B, LOC_F,
    PICKUP, DROPOFF, REFUEL, ACTION_NAMES, N_ACTIONS,
)
from taxi_visual import render_static, log_random, log_flat, log_hier, animate, slider
from IPython.display import HTML

EPISODE_STEPS = 240   # elke episode duurt precies 240 tijdstappen

def make_env(seed=42):
    return TaxiFuelEnv(
        seed=seed,
        customers_per_episode=9999,  # episode-einde via EPISODE_STEPS, niet via klanten
        max_fuel=20,
        reward_dropoff=50.0,
        reward_step=-1.0,
        reward_refuel=-10.0,
        reward_illegal=-10.0,
        reward_tow=-50.0,
        reward_lost_passenger=-50.0,
    )

env = make_env(42)
print(f"Aantal states : {env.n_states}")
print(f"Aantal acties : {env.n_actions}  ({', '.join(ACTION_NAMES)})")
env.reset()
print()
print(env.render())


### Visualisatie van de omgeving

Grafische render van de startpositie. **R** (rood), **G** (groen), **Y** (geel), **B** (blauw), **F** (paars) = tankstation. De **gele ster** is de bestemming; het **figuurtje** staat bij de pickup-locatie.


In [ ]:
env.reset()
fig, ax = render_static(env, figsize=(5, 5))
plt.show()


### Animatie 1 – Random agent (vóór enig leren)

Een volledig random agent kiest elke stap een willekeurige primitieve actie. Let op hoe chaotisch dit is: geen richting, geen doel, brandstof daalt willekeurig. Dit is de nulbaseline voor de volgende animaties.

*Rendering duurt ~5–10 seconden.*


In [ ]:
env_v1 = make_env(7)
frames_random = log_random(env_v1, n_steps=80, seed=0)

fig_r, ani_r = animate(frames_random, max_fuel=20,
                       title="Animatie 1: random agent", subsample=2)
plt.close(fig_r)
HTML(ani_r.to_jshtml())


**Begripscheck — beantwoord kort voor je verder gaat:**

* De state van onze taxi bestaat uit: positie (5×5 = 25 mogelijkheden), passagier-locatie (5 mogelijkheden: R/G/Y/B/in taxi), bestemming (4 mogelijkheden), en brandstof (0–20 = 21 niveaus). Hoeveel states zijn dat in totaal?
* We hebben 7 primitieve acties. Hoeveel state-actie paren moet een flat Q-learner leren?
* Stel elke state-actie combinatie wordt gemiddeld 1 keer bezocht per episode van 240 stappen. Hoeveel episodes zijn dan nodig om alles te zien?


## <font color='green'>Vul hier je antwoord in: </font>

---

## 1. Flat Q-learning

We beginnen met de standaard Q-learning aanpak: één Q-tabel over alle (state, primitieve actie) combinaties. De update-regel ken je uit Assignment 2:

$$Q(s, a) \leftarrow Q(s, a) + \alpha \bigl[ r + \gamma \max_{a'} Q(s', a') - Q(s, a) \bigr]$$

De drie termen hebben een duidelijke betekenis:
* $r$ = de directe reward die je net ontving
* $\gamma \max_{a'} Q(s', a')$ = geschatte toekomstige waarde vanuit de nieuwe state
* De update trekt $Q(s,a)$ een stapje richting dit "doelwit" (TD-target)

### Q1.a De epsilon-greedy strategie

Voordat je de update schrijft, eerst een begripscheck over de verkenningsstrategie. Welke exploratie strategie is hier geimplementeerd?

## <font color='green'>Vul hier je antwoord in: </font>

In [ ]:
def q_learn_flat(env, n_episodes=10000, alpha=0.1, gamma=0.95,
                 epsilon_start=1.0, epsilon_end=0.05, seed=0):
    """
    Flat tabular Q-learning op de Taxi-Fuel taak.
    Eén episode = EPISODE_STEPS tijdstappen.
    """
    rng = np.random.default_rng(seed)
    Q   = np.zeros((env.n_states, env.n_actions))
    rewards_per_ep = []

    for ep in range(n_episodes):
        epsilon = epsilon_start - (epsilon_start - epsilon_end) * (ep / n_episodes)
        s = env.reset()
        total_r    = 0.0
        step_count = 0

        while step_count < EPISODE_STEPS:

            if rng.random() < epsilon:
                a = int(rng.integers(0, N_ACTIONS))
            else:
                qs = Q[s]
                a  = int(rng.choice(np.flatnonzero(qs == np.max(qs))))

            s_next, r, _, _ = env.step(a)
            # ===== Q-LEARNING UPDATE: VUL ZELF IN =====
            # Hint: bereken eerst td_target = r + gamma * max Q(s_next)
            #       dan: Q[s, a] += alpha * (td_target - Q[s, a])

            # JOUW CODE HIER

            ############################################################

            s       = s_next
            total_r += r
            step_count += 1

        rewards_per_ep.append(total_r)

    return Q, rewards_per_ep


print("Training flat Q-learning (duurt ~25-30 seconden)...")
t0 = time.time()
Q_flat, rewards_flat = q_learn_flat(make_env(42), seed=42)
print(f"Klaar in {time.time()-t0:.1f}s  |  laatste 200 ep: {np.mean(rewards_flat[-200:]):.1f}")


In [ ]:
def smooth(x, window=100):
    return np.convolve(x, np.ones(window) / window, mode='valid')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(rewards_flat, alpha=0.15, color='C0')
ax.plot(np.arange(99, len(rewards_flat)), smooth(rewards_flat),
        color='C0', linewidth=2, label='Flat Q-learning (100-ep avg)')
ax.set_xlabel('Episode')
ax.set_ylabel('Totale reward per 240-stappen episode')
ax.set_title('Leercurve: Flat Q-learning (10 000 episodes)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print(f"Eerste 500 ep:  {np.mean(rewards_flat[:500]):.0f}")
print(f"Laatste 500 ep: {np.mean(rewards_flat[-500:]):.0f}")


### Animatie 2 – Flat Q na training

*Rendering duurt ~5–10 seconden.*


In [ ]:
env_v2 = make_env(7)
frames_flat = log_flat(env_v2, Q_flat, n_steps=240, seed=3)
print(f"Klanten afgeleverd: {env_v2.customers_served}")
fig_f, ani_f = animate(frames_flat, max_fuel=20,
                       title="Animatie 2: flat Q na 10 000 episodes", subsample=3)
plt.close(fig_f)
HTML(ani_f.to_jshtml())


### Q1.b Reflectie

* Hoe ontwikkelt de reward zich over 10 000 episodes? Noem een concreet getal   uit de leercurve (begin vs. einde).
* Bekijk animatie 2: heeft de agent een herkenbaar beleid? Beschrijf in één zin   wat hij lijkt te doen.
* Wat verwacht je dat hiërarchie zal verbeteren? Formuleer een verwachting.


## <font color='green'>Vul hier je antwoord in: </font>

---

## 2. Het Options Framework & pretraining

### Waarom flat Q vastloopt

Flat Q moet de waarde leren van elke (state, actie) combinatie. Met 10 500 states en 7 acties zijn dat **73 500 parameters**. Om te leren dat "rij naar bestemming → dropoff → +50" loont, moet de agent toevallig die hele sequentie voltooien. In 240 stappen per episode, verdeeld over een enorme state-ruimte, gebeurt dat te zelden voor stabiel leren.

### De oplossing: opties

**Sutton, Precup & Singh (1999)** definieerden een **optie** als een macro-actie met drie onderdelen:

$$o = (\mathcal{I},\, \pi,\, \beta)$$

| Onderdeel | Uitleg | Voorbeeld: GoToR |
|---|---|---|
| $\mathcal{I}$ | **Initiation set** — in welke states mag de optie starten? | Alle states (altijd mag je naar R gaan) |
| $\pi$ | **Option-policy** — welke primitieve actie in elke state? | Navigeer via kortste pad richting R |
| $\beta$ | **Terminatieconditie** — wanneer stopt de optie? | Zodra taxi op locatie R aankomt |

In dit werkcollege definiëren we 8 opties:
* **GoToR, GoToG, GoToY, GoToB, GoToF** — navigeer naar een locatie (meerdere stappen)
* **Pickup, Dropoff, Refuel** — 1-stap acties verpakt als optie

De GoTo-policies leren we **offline** via pretraining in een mini-navigatie-omgeving (25 posities × 4 bewegingsacties), daarna worden ze **bevroren**. De top-level agent leert alleen *wanneer* welke optie te kiezen — een veel kleinere leeruitdaging.

### Wat is een Semi-MDP?

Normaal Q-learning werkt in een **Markov Decision Process (MDP)**: elke stap duurt precies 1 tijdstap. Zodra we opties introduceren duurt elke "beslissing" een variabel aantal stappen (τ). Dat maakt het een **Semi-MDP (SMDP)**: beslissingen vinden niet op regelmatige tijdstippen plaats, maar zodra de vorige optie klaar is.

De Q-update moet daarvoor aangepast worden. In een gewone MDP: $r + \gamma \max Q(s')$. In een SMDP:

$$G + \gamma^{\tau} \max_{o'} Q(s', o')$$

* **G** is de *cumulatieve* reward die gedurende de hele optie-uitvoering is opgebouwd   (τ stappen lang)
* **$\gamma^{\tau}$** verdisconteert τ stappen in de toekomst in één keer — hoe   langer de optie duurde, hoe meer we verdisconteren

Het grote voordeel: de reward-informatie van een volledige rit (van F naar R, klant ophalen) wordt in **één update** teruggepropageerd naar de top-level beslissing. Bij flat Q zou dat over τ afzonderlijke updates moeten.


In [ ]:
options = get_default_options()
print("Beschikbare opties:")
for i, opt in enumerate(options):
    kind = "GoTo (geleerd)" if isinstance(opt, LearnedGoToOption) else "Primitief"
    print(f"  {i}: {opt.name:10s}  [{kind}]")



### Hoe werkt de pretraining?

Elke `LearnedGoToOption` heeft een eigen mini Q-tabel van **25 × 4**: 25 posities op het 5×5 grid en 4 bewegingsacties (N, Z, O, W). Pickup, Dropoff en Refuel bestaan niet in dit mini-probleem; de optie leert alleen navigeren naar één vaste locatie.

`pretrain_go_to_options()` traint elke GoTo-optie apart via 1500 korte episodes. Elke episode (`train_episode`) werkt als volgt:

1. **Startpositie willekeurig** gekozen op het 5×5 grid
2. **Epsilon-greedy actiekeuze** uit de 4 bewegingsacties (epsilon = 0.1)
3. **Rewards**:
   - Op doel aankomen: +20
   - Elke bewegingsstap: −1
   - Tegen een muur lopen: −2 (positie verandert niet)
4. **Q-update**: gewone TD-update — dezelfde formule als in blok 1, maar dan in de mini-omgeving
5. **Episode eindigt** zodra het doel bereikt is of na 30 stappen

Na 1500 episodes convergeert elke optie naar een vrijwel optimale navigatiepolicy. Je ziet dat aan de pretraining-output: de gemiddelde episode-lengte daalt van ~15 stappen (begin, grotendeels random) naar ~5 stappen (einde, near-optimal).

**Wat de pretraining niet bevat**: passagiers, brandstof, pickup/dropoff. De optie leert puur navigeren, onafhankelijk van de taxi-taak. Daarna wordt de policy **bevroren** en gebruikt door de top-level Q-learner.


In [ ]:
# Pretrain de GoTo-opties offline (duurt < 1 seconde)
print("Pretraining GoTo-opties...")
t0 = time.time()
pretrain_hist = pretrain_go_to_options(options, n_episodes_per_option=1500, seed=42)
print(f"Klaar in {time.time()-t0:.2f}s\n")
for name, lengths in pretrain_hist.items():
    avg_start = np.mean(lengths[:50])
    avg_end   = np.mean(lengths[-100:])
    print(f"  {name:8s}: {avg_start:.1f} stappen (begin) → {avg_end:.1f} stappen (einde)")


### Visualiseer de geleerde GoTo-policies

Na pretraining heeft elke GoTo-optie een Q-tabel (25 posities × 4 acties). De heatmap toont de waarde van de beste actie per cel; de pijlen geven de greedy richting. Controleer: wijzen de pijlen naar het juiste doel (ster)?


In [ ]:
def plot_option_policies(options):
    goto_opts = [o for o in options if isinstance(o, LearnedGoToOption)]
    n = len(goto_opts)
    fig, axes = plt.subplots(1, n, figsize=(3.5 * n, 3.8))
    arrow = {0: '↓', 1: '↑', 2: '→', 3: '←'}

    for ax, opt in zip(axes, goto_opts):
        V = np.max(opt.Q, axis=1).reshape(5, 5)
        G = np.argmax(opt.Q, axis=1).reshape(5, 5)
        im = ax.imshow(V, origin='upper', cmap='viridis')
        vmin, vmax = V.min(), V.max()
        thr = vmin + 0.55 * (vmax - vmin + 1e-8)
        for r in range(5):
            for c in range(5):
                color = 'white' if V[r, c] > thr else 'black'
                ax.text(c, r, arrow[int(G[r, c])],
                        ha='center', va='center', fontsize=16,
                        color=color, fontweight='bold')
        tr, tc = opt.target
        ax.scatter(tc, tr, s=300, marker='*', edgecolor='black',
                   linewidth=1, zorder=5, color='gold')
        ax.set_title(opt.name, fontsize=11)
        ax.set_xticks([]); ax.set_yticks([])

    fig.suptitle("Greedy policies van de GoTo-opties (gouden ster = doel)", y=1.01)
    plt.tight_layout(); plt.show()

plot_option_policies(options)


In [ ]:
# Demo: voer GoToR uit in de echte omgeving
env_demo = make_env(42)
env_demo.reset()
print("Vóór GoToR:")
print(env_demo.render())
s_next, G, done, tau = options[0].execute(env_demo)
print(f"\nNá GoToR  |  stappen: {tau}  |  reward: {G:.1f}")
print(env_demo.render())


### Animatie 3 – Na pretraining, vóór top-level leren

De GoTo-policies zijn nu gepretrained maar de **top-level Q-tabel is nul**. De agent kiest opties willekeurig (random tiebreak). Navigatie *binnen* een optie is al coherent; de keuze *welke* optie te nemen is nog willekeurig.

Vergelijk met animatie 1: wat is er verbeterd?

*Rendering duurt ~5 seconden.*


In [ ]:
Q_zero    = np.zeros((env.n_states, len(options)))
env_v3    = make_env(7)
frames_mid = log_hier(env_v3, Q_zero, options, n_steps=100, seed=1)

fig_m, ani_m = animate(frames_mid, max_fuel=20,
                       title="Animatie 3: opties gepretrained, top-level Q=0",
                       subsample=2)
plt.close(fig_m)
HTML(ani_m.to_jshtml())


---

## 3. Hiërarchische Q-learning

Nu leren we Q-waardes op het niveau van opties. De SMDP Q-update:

$$Q(s, o) \leftarrow Q(s, o) + \alpha \bigl[ G + \gamma^\tau \max_{o'} Q(s', o') - Q(s, o) \bigr]$$

Vergelijk met de gewone Q-update die je zonet gebruikte:

| Gewone Q (MDP) | SMDP Q (opties) |
|---|---|
| $r$ (reward van 1 stap) | $G$ (cumulatieve reward van de hele optie) |
| $\gamma$ (1 stap verdisconteren) | $\gamma^\tau$ (τ stappen verdisconteren) |
| $\max_{a'} Q(s', a')$ | $\max_{o'} Q(s', o')$ (over opties, niet acties) |

### Q3.a Schrijf de SMDP Q-update

Gebruik `option.execute(env)` om een optie uit te voeren. Die geeft $(s', G, \text{done}, \tau)$ terug.


In [ ]:
def q_learn_hierarchical(env, options, n_episodes=10000, alpha=0.1, gamma=0.95,
                         epsilon_start=1.0, epsilon_end=0.02, seed=0):
    """
    SMDP Q-learning over opties. GoTo-policies zijn bevroren.
    Eén episode = EPISODE_STEPS tijdstappen.
    """
    rng    = np.random.default_rng(seed)
    n_opts = len(options)
    Q      = np.zeros((env.n_states, n_opts))
    rewards_per_ep = []

    for ep in range(n_episodes):
        epsilon = epsilon_start - (epsilon_start - epsilon_end) * (ep / n_episodes)
        s = env.reset()
        total_r    = 0.0
        step_count = 0

        while step_count < EPISODE_STEPS:
            if rng.random() < epsilon:
                o_idx = int(rng.integers(0, n_opts))
            else:
                qs    = Q[s]
                o_idx = int(rng.choice(np.flatnonzero(qs == np.max(qs))))
            opt = options[o_idx]

            s_start      = s
            s, G, _, tau = opt.execute(env)
            step_count  += tau
            # ===== SMDP Q-UPDATE: VUL ZELF IN =====
            # G   = cumulatieve reward tijdens de optie
            # tau = aantal primitieve stappen dat de optie duurde
            # Hint: de update-formule staat in de tabel hierboven.

            # JOUW CODE HIER


            # ======================================

            total_r += G

        rewards_per_ep.append(total_r)

    return Q, rewards_per_ep


print("Training hierarchical Q-learning (duurt ~15-20 seconden)...")
t0 = time.time()
Q_hier, rewards_hier = q_learn_hierarchical(make_env(42), options, seed=42)
print(f"Klaar in {time.time()-t0:.1f}s  |  laatste 200 ep: {np.mean(rewards_hier[-200:]):.1f}")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(np.arange(99, len(rewards_flat)), smooth(rewards_flat),
        color='C0', linewidth=2, label='Flat Q-learning')
ax.plot(np.arange(99, len(rewards_hier)), smooth(rewards_hier),
        color='C1', linewidth=2, label='Hierarchical Q-learning')
ax.set_xlabel('Episode')
ax.set_ylabel('Totale reward (100-ep gemiddelde)')
ax.set_title('Flat vs. Hiërarchische Q-learning — 10 000 episodes')
ax.legend(loc='lower right'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f"Flat Q,         laatste 500 ep: {np.mean(rewards_flat[-500:]):.1f}")
print(f"Hierarchical Q, laatste 500 ep: {np.mean(rewards_hier[-500:]):.1f}")


### Animatie 4 – Hiërarchische agent na training

De agent is volledig getraind. Let op:
* Rijdt hij efficiënte routes?
* Tankt hij **proactief** (voordat de tank leeg is) of pas als het lampje brandt?
* Hoeveel klanten haalt hij in 240 stappen?

Vergelijk met animatie 1 (random), 2 (flat) en 3 (midway).

*Rendering duurt ~10–15 seconden.*


In [ ]:
env_v4 = make_env(0)
frames_trained = log_hier(env_v4, Q_hier, options, n_steps=240, seed=0)
print(f"Klanten afgeleverd: {env_v4.customers_served} in {len(frames_trained)-1} stappen")
fig_t, ani_t = animate(frames_trained, max_fuel=20,
                       title="Animatie 4: getrainde hiërarchische agent",
                       subsample=4)
plt.close(fig_t)
HTML(ani_t.to_jshtml())


### Q3.b Reflectie op de vergelijking flat & hierachische Q-learning

* **Bereken**:De top-level Q-tabel kiest tussen opties, niet tussen primitieve acties. Hoeveel opties hebben we? En hoeveel state-optie paren moet de top-level Q-tabel leren?
* **Bereken**:Ga ervan uit dat een optie gemiddeld 4–8 primitieve stappen duurt. Een episode duurt 240 stappen. Hoeveel top-level updates doet de hiërarchische agent per episode? Vergelijk dit met flat Q (240 updates).
* **Bedenk**: de hiërarchische Q-tabel heeft meer state-actie paren dan flat Q, én doet minder updates per episode. Waarom leert hij dan toch sneller?

* **Begripsvraag**: Stel tau = 6 (de GoToR optie duurde 6 stappen) en gamma = 0.95.   Wat is de waarde van gamma^tau? Wat betekent dit voor hoe de agent toekomstige beloningen waardeert? Hoe beinvloed dit zijn strategie?


## <font color='green'>Vul hier je antwoord in: </font>

---

## 4. Brandstof-analyse: homeostase of allostase?

### Wat zijn homeostase en allostase?

**Homeostase** is het klassieke concept uit de fysiologie (Cannon, 1932): het lichaam herstelt verstoringen via negatieve feedback. Denk aan een thermostaat: de verwarming gaat aan *zodra* het te koud wordt, en uit *zodra* de gewenste temperatuur bereikt is. Het systeem reageert op een tekort dat al opgetreden is.

**Allostase** is een uitbreiding die Sterling (2012) en Schulkin & Sterling (2019) voorstellen: het brein reguleert proactief en voorspellend. In plaats van te wachten tot er een tekort is, anticipeert het op toekomstige behoeften. Een dier dat weet dat de winter komt, eet in de herfst extra — lang voordat er voedselgebrek is.

Een scherper voorbeeld: een marathonloper drinkt *tijdens* de race water, niet pas als hij al uitgedroogd is. Die timing is cruciaal: als je wacht op het tekort, is het al te laat voor optimale prestaties.

### Wat voorspelt het allostatische framework voor onze taxi?

De taxi heeft een "interne state" — zijn brandstof. Een homeostatische taxi tankt pas als de tank bijna leeg is. Een allostatische taxi anticipeert: hij tankt op het moment dat het strategisch het beste uitkomt, ook als de tank nog redelijk vol is — namelijk vóórdat hij aan een lange rit begint waarbij hij anders strandt.

Keramati & Gutkin (2014, *eLife*) lieten formeel zien dat reward-maximalisatie wiskundig equivalent is aan fysiologische stabilisatie: een agent die optimaal leert gedraagt zich automatisch allostatisch. Laten we kijken of dat in ons geval ook zo is voor Flat and Hierachical Q.


In [ ]:
def get_fuel_trace(Q, use_hier, seed=0):
    env_e = make_env(seed)
    s     = env_e.reset()
    rng   = np.random.default_rng(seed)
    fuel  = [env_e.fuel]
    sc    = 0
    while sc < 240:
        if use_hier:
            o          = int(rng.choice(np.flatnonzero(Q[s] == np.max(Q[s]))))
            s, G, _, tau = options[o].execute(env_e)
            sc        += tau
        else:
            a          = int(rng.choice(np.flatnonzero(Q[s] == np.max(Q[s]))))
            s, r, _, _ = env_e.step(a)
            sc        += 1
        fuel.append(env_e.fuel)
    return fuel, env_e.customers_served

flat_fuel, flat_served = get_fuel_trace(Q_flat, use_hier=False, seed=0)
hier_fuel, hier_served = get_fuel_trace(Q_hier, use_hier=True,  seed=0)

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=False)
for ax, trace, label, served, color in [
    (axes[0], flat_fuel, "Flat Q-learning",         flat_served, 'C0'),
    (axes[1], hier_fuel, "Hierarchical Q-learning",  hier_served, 'C1'),
]:
    ax.plot(trace, color=color, linewidth=1.5)
    ax.axhline(0, color='red', ls='--', alpha=0.5, label='Tank leeg (sleep)')
    ax.set_ylabel("Brandstof"); ax.set_ylim(-2, 22)
    ax.set_title(f"{label}  —  {served} klanten in 240 stappen")
    ax.grid(True, alpha=0.3); ax.legend(loc='lower right', fontsize=9)
axes[1].set_xlabel("Stap")
plt.tight_layout(); plt.show()

for trace, label in [(flat_fuel, "flat"), (hier_fuel, "hier")]:
    refuel_levels = [trace[i-1] for i in range(1, len(trace)) if trace[i] > trace[i-1]]
    if refuel_levels:
        print(f"{label}: tankt bij gemiddeld brandstof = {np.mean(refuel_levels):.1f} "
              f"(min={min(refuel_levels)}, max={max(refuel_levels)}, n={len(refuel_levels)}×)")
    else:
        print(f"{label}: nooit getankt")


### Q4.a Reflectie — homeostase of allostase?

* Op welk gemiddeld brandstofniveau tankt de hiërarchische agent?   Vergelijk dit met de flat Q-agent.
* Is het gedrag van de hiërarchische agent homeostatisch of allostatisch?   Gebruik de thermostaat-analogie in je antwoord.
* Sterling (2012) stelt dat allostase vereist dat het systeem een *model*   heeft van toekomstige behoeften. Welke informatie in de state van onze   taxi maakt allostatisch gedrag mogelijk?
* Bedenk een menselijk voorbeeld waarbij homeostatische regulatie zou falen   maar allostatische regulatie zou slagen. Leg uit waarom.


## <font color='green'>Vul hier je antwoord in: </font>

---

## 5. Transfer en latent leren (BONUS)

### Tolman en de cognitieve kaart

In het college bespraken we het experiment van Edward Tolman(Tolman, 1948, *Psychological Review*) met ratten in een doolhof. Een groep ratten werd dagelijks door het doolhof geleid zonder enige beloning — ze leken niets te leren. Maar toen er voedsel aan het einde werd geplaatst, bereikten deze ratten het voedsel bijna direct, sneller dan ratten die altijd al beloning hadden gekregen.

Tolmans conclusie: de ratten hadden tijdens de verkenning een **cognitieve kaart** opgebouwd — een interne representatie van de ruimte — ook zonder directe beloning. Dit noemde hij **latent leren**: kennis die niet direct in gedrag zichtbaar is, maar beschikbaar is zodra die kennis relevant wordt.

### Wat heeft onze agent "latent geleerd"?

Onze hiërarchische agent heeft tijdens pretraining GoTo-policies geleerd: interne representaties van de ruimte voor elke locatie. Die kennis zit in de Q-tabel van elke optie. Bij de top-level training is deze ruimtelijke kennis al aanwezig — net zoals Tolmans ratten hun cognitieve kaart al hadden.

Nu de vraag: wat gebeurt er als de omgeving verandert?


### Q5 Reflectie — transfer en latent leren

* Stel we veranderen een paar muren en laten FlatQ en OptionsQ hier in rondrijden, hoe presteren de agents na de wegversperring?
* Welk deel van de kennis van de hiërarchische agent is **kwetsbaar** voor de nieuwe muur, en welk deel blijft geldig?
* Tolmans ratten leerden de ruimte kennen *zonder* beloningen. Op welke manier lijkt de pretraining van onze GoTo-opties op latent leren? Waarin verschilt het?


## <font color='green'>Vul hier je antwoord in: </font>

---

## Wrap-up

Wat je nu hebt gedaan:

1. **Flat Q-learning** getraind (10 000 episodes) en gezien dat het leert maar langzamer dan hiërarchie.
2. **GoTo-opties gepretrained** via offline navigatie-leren —
3. **Hiërarchische Q-learning** geïmplementeerd met de SMDP-update en gezien dat het dramatisch sneller convergeert.
4. **Brandstofbeheer** geanalyseerd via de homeostase/allostase lens.
5. **Transfer** getest via een wegversperring, en de link gelegd met Tolmans latent leren.

Bewaar je plots en notities voor de discussiesessie:
* **Discussievraag** — leercurve: twee mechanismen voor het verschil.
* **Discussievraag** — transfer: welke kennis blijft herbruikbaar?
* **Discussievraag** — allostase: wanneer tankt de agent en wat voorspelt het allostatische framework?
* **Discussievraag** — option learning: hoe had de agent opties zelf kunnen ontdekken, en wat zegt dat over sociaal leren?
